In [ ]:
import pandas as pd 
import numpy as np 
import re
from tqdm import tqdm
import matplotlib.pyplot as plt
from datetime import datetime

import pandas as pd
import os

In [ ]:
from IPython.display import clear_output

In [ ]:
%run ./methods/methods.py
clear_output()

In [ ]:
date = datetime.today().strftime('%Y%m%d')

### 1. Import data

In [ ]:
origin_path = "H:/projet/data/patients_FR/"
path_to_file = os.path.join(origin_path,'patients_FR_geom_clinic_socioeco.csv')

df_patients = pd.read_csv(path_to_file,sep=";",dtype=str).drop('Unnamed: 0',axis=1)

##Normalisation des colonnes 
df_patients = df_patients.rename({'adresse':'RUE', 
                                  'codepost':'CPVILLE', 
                                  'nom_commune_postal': 'VILLE'},axis=1)

df_odonyme = pd.read_csv("./data/odonymes.txt",delimiter="|")
liste_odonyme = pd.Series(df_odonyme['synonym'].str.upper())

df_prenom = pd.read_csv('./data/Prenoms.csv',encoding='utf-8',sep=';')
df_prenom= df_prenom.dropna(subset='01_prenom')
liste_prenom = pd.Series(df_prenom['01_prenom'])

### 2. Data standardization
raw variables from our database :

- RUE: street number, street and street name
- CPVILLE: postcode
- VILLE : town
  
result variables from geocoding :
- result_housenumber: street number
- result_name: lane and lane name
- pc_city : postcode
- city : city 

In [ ]:
## Only data from mainland France is kept 
mask = (
    df_patients['CPVILLE'].str.startswith('97') | 
    df_patients['CPVILLE'].str.startswith('98') |
    df_patients['CPVILLE'].str.startswith('99'))
df = df_patients[~mask] 

## Lines that do not contain one or other of the following are deleted 
df = df.dropna(subset=['RUE','street'])


df = df.rename({'RUE':'adr_init',
                'CPVILLE': 'cp_init',
                'VILLE': 'ville_init',
                'street': 'adr_geo',
                'pc_city':'cp_geo',
                'city': 'ville_geo'}, axis=1)

## Correction of common errors in postal addresses
df_cleaned_rue = normalisation_adresse(df,'adr_init')
df_cleaned = normalisation_adresse(df_cleaned_rue,'adr_geo')

## Standardisation of addresses - replacement of lane abbreviations
df_ad = remplacer_types_de_voies(df_cleaned, 'adr_init', df_odonyme)
df_final = remplacer_types_de_voies(df_ad, 'adr_geo', df_odonyme)

### 3. Text data alignement

In [ ]:
df_chunks = df_final.copy().reset_index(drop=True)
chunk_size = 1000

df_final_chunks = {}

if not os.path.exists('./data/outputs'): 
    os.mkdir('./data/outputs')

for j, chunk in tqdm(enumerate(range(0, len(df_chunks), chunk_size))): 
    df_chunk = df_chunks.iloc[chunk:chunk + chunk_size] 
    
    for i in tqdm(df_chunk.index) : 
    
        ad_brute = df_chunk.loc[i,'adr_init']
        ad_geo = df_chunk.loc[i,'adr_geo']

        char_alignment_score, token_alignment_score, brute_aligned, geo_aligned, unmatched_brute, unmatched_geo = Needleman_Wunch_update(ad_brute,ad_geo,return_all=True)

        if pd.isna(char_alignment_score) or pd.isna(token_alignment_score) : 
            print(f'Error aligning for chunk {chunk}, at row {i}')
        else  :

            metrics = calculate_alignment_metrics(brute_aligned, geo_aligned)
            df_chunk.loc[i,'score'] =  metrics['alignment_score']
            df_chunk.loc[i,'char_alignment_score'] =  char_alignment_score
            df_chunk.loc[i,'token_alignment_score'] =  token_alignment_score

        if unmatched_brute :
            df_chunk.loc[i,'not_matched_brute'] =  ' '.join(unmatched_brute)

        if len(unmatched_geo)>=1 : 
            df_chunk.loc[i,'not_matched_geo'] =  ' '.join(unmatched_geo)
            clear_output()
            
    if j ==0 : 
        aligned_df = df_chunk
    else : 
        aligned_df = pd.concat([aligned_df,df_chunk],axis=0)
        aligned_df.to_csv(os.path.join(origin_path,f'patients_geocoded_aligned_{date}.csv'))
        aligned_df.to_csv(f'./data/outputs/patients_geocoded_aligned_{date}.csv')   ## Change to your filepath  

aligned_df.to_csv(os.path.join(origin_path,'patients_geocoded_aligned_{date}.csv'))    ## Change to your filepath  
aligned_df.to_csv(f'./data/outputs/patients_geocoded_aligned_{date}.csv')


In [ ]:
directory_path = './data/outputs/'
for file in os.listdir(directory_path):
    if not 'wstreetInfos' in file:
        filepath = os.path.join(directory_path, file)

df = pd.read_csv(filepath,dtype=str).drop('Unnamed: 0',axis=1)


### 4. Identification of road type position

In [ ]:
df = find_pos_elem(df, 'adr_init',liste_odonyme,'street_type','contains_street_type','pos_street_type',drop_col_elem=False)

In [ ]:
##INIT

## Analyse plus poussée pour les erreurs de typo (voie associé à nom de rue)


df_filtre = df[df['contains_street_type']==False]
pattern = r"\b(AVENUE|RUE|PLACE|IMPASSE|BOULEVARD|ALLEE|SQUARE|ROUTE|ESPLANADE|CHEMIN|GRANDE\sRUE|ROND\sPOINT|FAUBOURG|JARDIN|VIA|GALERIE|VOIE|QUAI|PASSAGE|COUR|COURS|CITE|PARVIS|HAMEAU|VILLA|VILLAGE|VILLE|TERRASSE|PROMENADE|SENTIER|CARREFOUR|CHAUSSEE|DOMAINE|CLOS|MOULIN|CENTRE|MAIL|BOIS|PROMENEE|VALLEE|RESIDENCE|QUARTIER|LOTISSEMENT|TRAVERSE|LIEU\sDIT|FERME|LE\sBOURG|PARC|COTE)[A-Za-z]+"

df_filtre['contains_street_type'] = df_filtre['adr_init'].str.contains(pattern, regex=True)

df.loc[df_filtre.index,'contains_street_type'] = df_filtre['contains_street_type'].values


pattern2 = r"(.*)(AVENUE|RUE|PLACE|IMPASSE|BOULEVARD|ALLEE|SQUARE|ROUTE|ESPLANADE|CHEMIN|GRANDE\sRUE|ROND\sPOINT|FAUBOURG|JARDIN|VIA|GALERIE|VOIE|QUAI|PASSAGE|COUR|COURS|CITE|PARVIS|HAMEAU|VILLA|VILLAGE|VILLE|TERRASSE|PROMENADE|SENTIER|CARREFOUR|CHAUSSEE|DOMAINE|CLOS|MOULIN|CENTRE|MAIL|BOIS|PROMENEE|VALLEE|RESIDENCE|QUARTIER|LOTISSEMENT|TRAVERSE|LIEU\sDIT|FERME|LE\sBOURG|PARC|COTE)(.*)"

df_filtre[['grp1','grp2','grp3']]=df_filtre.loc[:, 'adr_init'].str.extract(pattern2, expand=True)

df_filtre['adr_cleaned'] = df_filtre['grp1'] + " " + df_filtre['grp2'] + " "+ df_filtre['grp3']

df_filtre_valid = df_filtre.dropna(subset=['adr_cleaned']).copy()
df_filtre_valid = df_filtre_valid[['adr_cleaned']].set_index(df_filtre_valid.index)


df_filtre = df_filtre.dropna(subset=['adr_cleaned'])

df.loc[df_filtre_valid.index, 'adr_init'] = df_filtre_valid['adr_cleaned']

df = find_pos_elem(df, 'adr_init',liste_odonyme,'street_type','contains_street_type','pos_street_type',drop_col_elem=False)
clear_output()



In [ ]:
df.to_csv(f"./data/outputs/patients_geocoded_aligned_{date}_wstreetInfos.csv",sep=";")

In [ ]:
for file in os.listdir(directory_path):
    if 'wstreetInfos' in file:
        filepath = os.path.join(directory_path, file)

df = pd.read_csv(filepath,sep=";").drop('Unnamed: 0',axis=1)

clear_output()

### 5. Addresses differentiated by composition

In [ ]:
df_w_street = df[df['contains_street_type']==True]
df_wout_street = df[df['contains_street_type']==False]

### 6. Identification of common noises by type of address

In [ ]:
liste_prenom = liste_prenom.str.replace(r'[^a-zA-Z0-9\s]', '', regex= True)
liste_prenom = liste_prenom.str.replace(r'\d', '', regex= True)
liste_prenom = liste_prenom.str.strip()
liste_prenom = liste_prenom.str.upper() 

liste_bister = pd.Series(['BIS','TER','QUATER'])
liste_common_elements = pd.Series(['LE','LA','LES','A','AUX','DE','DU','DES','L','D','E'])

## concaténation de l'ensemble des éléments
to_clean = pd.concat([liste_prenom,liste_odonyme,liste_bister,liste_common_elements])


bruit_vrais_w_street = find_most_common_biaises(df_w_street,to_clean)
bruit_vrais_wout_street = find_most_common_biaises(df_wout_street,to_clean)

## On considère qu'un bruit a au moins 2 occurences 
# bruit_vrais_w_street = bruit_vrais_w_street[bruit_vrais_w_street['count']>2]
# bruit_vrais_wout_street = bruit_vrais_wout_street[bruit_vrais_wout_street['count']>2]

clear_output()

### 7. Filter address elements considered to be biased

From the list of ‘biases’, manually identify those resulting from a geocoding error or an alignment error to find the most relevant ones.  

use the function filter_by_word_in_column(df, column_name, word)

In [ ]:
## On parcourt uniquement les adresses ayant des éléments non alignés 

df_bruite_w_street = df_w_street[(~df_w_street['not_matched_brute'].isna())&(df_w_street['not_matched_brute']!="")]

## Elements récurrents générants des erreurs dans le géocoage 
not_a_biais_align = ['CHAMPS','GAUTHIER','F','N','ROCHEFOUCAULT','VANNEAU','MANSARD','FREDERIQUE','ROCHECHOUARD','QUATRES','MARBOEUF','CHEVREUIL','RENAND',
                    'APPOLINAIRE','COMETTE','ALLOUETTES','DESFORGES','BRILLANT','GUTTENBERG','GUTTEMBERG','GALIENNI','GAULLES','SEMART','GOERGES','RIQUETTI',
                    'BOLIVARD','BALLARD','SALVATOR','DOLLET','EDMONT','FRANCKLIN','THIMBAUD','BOULLAINVILLIERS','CAULINCOURT','SKOBSTOV','TIMBAULT','SALLENGRO',
                    'LONCHAMPS','ABARTHOLOME','TILLEUILS']
not_a_biais_geoc = ['PERI','FOCH','CONTENTIN','NOVEMBRE','MUTUALISTE','REUNION','DANREMONT','LADWIG','RPC']

##erreur de sensibilité alignement 
not_a_biais = not_a_biais_align + not_a_biais_geoc
## erreur géocodage 
resultat_w_street = {}
for mot in bruit_vrais_w_street.head(100).index:
    if mot not in not_a_biais : 
    # Compter le nombre de lignes contenant le mot
        resultat_w_street[mot] = df_bruite_w_street["not_matched_brute"].str.contains(fr"\b{mot}\b", regex=True).sum()



resultat_w_street_df = pd.DataFrame(list(resultat_w_street.items()), columns=["Mot", "Occurrences"])


freq_cumul_w_street = resultat_w_street_df.sort_values(by='Occurrences',ascending=True)

freq_cumul_w_street['freq_cumulee'] = freq_cumul_w_street['Occurrences'].cumsum()
freq_cumul_w_street['prop_cumul'] = freq_cumul_w_street['freq_cumulee'] / freq_cumul_w_street['Occurrences'].sum()

freq_couv_w_street = freq_cumul_w_street.sort_values(by="Occurrences",ascending=False)

In [ ]:
biaises = list(freq_couv_w_street.Mot.head(10))

biaises_to_DF = pd.DataFrame(biaises , index = np.arange(len(biaises)), columns=['biais'])

biaises_to_DF.to_csv('./data/biaises_identified.csv',sep="|", columns=["biais"])


In [ ]:
# freq_couv_w_street = freq_couv_w_street.reset_index()
freq_couv_w_street[freq_couv_w_street['Mot']=='PREMIER']